# Fishnet Grid Generator

Generates a fishnet of rectangular cells over a bounding box defined by two lat/lon corner pairs.

## Two methods
| Method | Behaviour |
|--------|-----------|
| `"equal"` | Divides the bounding box width and height by `grid_size`, takes the **ceiling** to get column/row counts, then derives uniform cell dimensions — every cell is the same size. |
| `"true"` | Uses `grid_size` directly as the cell width and height in metres. Remainder (smaller) cells appear at the far edges. |

The lat/lon → metre conversion is estimated at the centre latitude of the bounding box using the standard geodetic series expansion (no external dependencies required).

In [3]:
import math
from typing import Literal


# ---------------------------------------------------------------------------
# Geodetic helper
# ---------------------------------------------------------------------------

def _deg_to_meters(lat_center_deg: float) -> tuple[float, float]:
    """
    Estimate metres per degree of latitude and longitude at the given latitude.

    Uses the standard geodetic series expansion — no external dependencies.

    Returns
    -------
    (m_per_deg_lat, m_per_deg_lon)
    """
    lat_rad = math.radians(lat_center_deg)

    m_per_lat = (
        111_132.92
        - 559.82  * math.cos(2 * lat_rad)
        + 1.175   * math.cos(4 * lat_rad)
        - 0.0023  * math.cos(6 * lat_rad)
    )
    m_per_lon = (
        111_412.84 * math.cos(lat_rad)
        - 93.5     * math.cos(3 * lat_rad)
        + 0.118    * math.cos(5 * lat_rad)
    )

    return m_per_lat, m_per_lon


# ---------------------------------------------------------------------------
# Main function
# ---------------------------------------------------------------------------

def create_fishnet(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float,
    grid_size: float,
    method: Literal["equal", "true"] = "equal",
) -> list[dict]:
    """
    Create a fishnet grid of rectangles over a bounding box.

    Parameters
    ----------
    lat1, lon1 : float
        One corner of the bounding box (any corner).
    lat2, lon2 : float
        The diagonally opposite corner of the bounding box.
    grid_size : float
        Target cell size in metres.
    method : {"equal", "true"}
        "equal" -- Divide total width and height by grid_size, take the
                   ceiling to get column/row counts, then compute uniform
                   cell dimensions so every cell is exactly the same size.
        "true"  -- Use grid_size directly for cell width and height.
                   Smaller remainder cells appear at the far (east/north)
                   edges of the bounding box.

    Returns
    -------
    list[dict]
        Each dict contains:
            "corners"  : list of 4 [lat, lon] pairs in
                         SW -> SE -> NE -> NW order.
            "centroid" : [lat, lon] of the cell centre.
    """
    # Normalise so lat_min < lat_max, lon_min < lon_max
    lat_min, lat_max = min(lat1, lat2), max(lat1, lat2)
    lon_min, lon_max = min(lon1, lon2), max(lon1, lon2)

    lat_center = (lat_min + lat_max) / 2
    m_per_lat, m_per_lon = _deg_to_meters(lat_center)

    total_width_m  = (lon_max - lon_min) * m_per_lon
    total_height_m = (lat_max - lat_min) * m_per_lat

    cells: list[dict] = []

    if method == "equal":
        # Ceiling division to get cell counts, then derive uniform cell size
        n_cols = math.ceil(total_width_m  / grid_size)
        n_rows = math.ceil(total_height_m / grid_size)

        cell_w_deg = (lon_max - lon_min) / n_cols
        cell_h_deg = (lat_max - lat_min) / n_rows

        for row in range(n_rows):
            for col in range(n_cols):
                sw_lat = lat_min + row * cell_h_deg
                sw_lon = lon_min + col * cell_w_deg
                ne_lat = sw_lat + cell_h_deg
                ne_lon = sw_lon + cell_w_deg

                cells.append({
                    "corners": [
                        [sw_lat, sw_lon],   # SW
                        [sw_lat, ne_lon],   # SE
                        [ne_lat, ne_lon],   # NE
                        [ne_lat, sw_lon],   # NW
                    ],
                    "centroid": [(sw_lat + ne_lat) / 2, (sw_lon + ne_lon) / 2],
                })

    elif method == "true":
        # Fixed cell size in metres; remainder (smaller) cells at far edges
        cell_w_deg = grid_size / m_per_lon
        cell_h_deg = grid_size / m_per_lat

        lat = lat_min
        while lat < lat_max - 1e-10:
            ne_lat = min(lat + cell_h_deg, lat_max)
            lon = lon_min
            while lon < lon_max - 1e-10:
                ne_lon = min(lon + cell_w_deg, lon_max)

                cells.append({
                    "corners": [
                        [lat,    lon   ],   # SW
                        [lat,    ne_lon],   # SE
                        [ne_lat, ne_lon],   # NE
                        [ne_lat, lon   ],   # NW
                    ],
                    "centroid": [(lat + ne_lat) / 2, (lon + ne_lon) / 2],
                })

                lon += cell_w_deg
            lat += cell_h_deg

    else:
        raise ValueError(f"method must be 'equal' or 'true', got {method!r}")

    return cells


In [4]:
# ---------------------------------------------------------------------------
# Example: Central London bounding box, 500 m grid
# ---------------------------------------------------------------------------

# SW corner: ~Waterloo, NE corner: ~King's Cross
LAT1, LON1 = 51.498, -0.113   # approx SW
LAT2, LON2 = 51.531,  0.002   # approx NE
GRID_M = 500                   # 500 m target cell size

# --- Method: equal (all cells same size) ---
cells_equal = create_fishnet(LAT1, LON1, LAT2, LON2, grid_size=GRID_M, method="equal")
print(f"[equal]  cells: {len(cells_equal)}")
print(f"         first cell corners: {cells_equal[0]['corners']}")
print(f"         first cell centroid: {cells_equal[0]['centroid']}")

# --- Method: true (fixed 500 m cells; remainder cells at far edge) ---
cells_true = create_fishnet(LAT1, LON1, LAT2, LON2, grid_size=GRID_M, method="true")
print(f"\n[true]   cells: {len(cells_true)}")
print(f"         first cell corners: {cells_true[0]['corners']}")
print(f"         first cell centroid: {cells_true[0]['centroid']}")

# --- Quick sanity: cell sizes in metres ---
m_lat, m_lon = _deg_to_meters((LAT1 + LAT2) / 2)

eq0 = cells_equal[0]["corners"]
eq_w = abs(eq0[1][1] - eq0[0][1]) * m_lon
eq_h = abs(eq0[2][0] - eq0[0][0]) * m_lat
print(f"\n[equal]  approx cell size: {eq_w:.1f} m (W) x {eq_h:.1f} m (H)")

tr0 = cells_true[0]["corners"]
tr_w = abs(tr0[1][1] - tr0[0][1]) * m_lon
tr_h = abs(tr0[2][0] - tr0[0][0]) * m_lat
print(f"[true]   interior cell size: {tr_w:.1f} m (W) x {tr_h:.1f} m (H)")


[equal]  cells: 128
         first cell corners: [[51.498, -0.113], [51.498, -0.1058125], [51.502125, -0.1058125], [51.502125, -0.113]]
         first cell centroid: [51.5000625, -0.10940625000000001]

[true]   cells: 128
         first cell corners: [[51.498, -0.113], [51.498, -0.10579730411353157], [51.50249405597982, -0.10579730411353157], [51.50249405597982, -0.113]]
         first cell centroid: [51.50024702798991, -0.10939865205676579]

[equal]  approx cell size: 498.9 m (W) x 458.9 m (H)
[true]   interior cell size: 500.0 m (W) x 500.0 m (H)
